# Hybrid RAG Pipeline
Combining **Dense Retrieval** (HuggingFace Embeddings) + **Sparse Retrieval** (BM25) with **Pinecone** and **Google Gemini** generation.

In [1]:
import os
import glob
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from dotenv import load_dotenv
from pypdf import PdfReader
from pinecone import Pinecone, ServerlessSpec
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone_text.sparse import BM25Encoder
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_google_genai import ChatGoogleGenerativeAI

print("All libraries imported successfully!")

d:\RAG PIPELINE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully!


In [3]:
load_dotenv()
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
HF_TOKEN = os.environ.get("HF_TOKEN")
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
INDEX_NAME = "hybrid-search-langchain-pinecone"

# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# Create index if not already present
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created index: {INDEX_NAME}")
else:
    print(f"Index '{INDEX_NAME}' already exists.")

# Connect to the index
index = pc.Index(INDEX_NAME)
print("Index status:", index.describe_index_stats())

Index 'hybrid-search-langchain-pinecone' already exists.
Index status: {'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'': {'vector_count': 8522}},
 'total_vector_count': 8522,
 'vector_type': 'dense'}


In [4]:
def clean_text(text: str) -> str:
    # Strip characters that cannot be represented in UTF-8
    return text.encode("utf-8", "ignore").decode("utf-8")

PDF_FOLDER = r"D:\RAG PIPELINE\AI knowledge"
PDF_PATHS = glob.glob(os.path.join(PDF_FOLDER, "*.pdf"))

print(f"Found {len(PDF_PATHS)} PDFs:")
for p in PDF_PATHS:
    print(" -", os.path.basename(p))

documents = []
for path in PDF_PATHS:
    reader = PdfReader(path)
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            text = clean_text(text)
            documents.append({
                "text": text,
                "source": os.path.basename(path),
                "page": page_num + 1,
            })

print(f"Extracted {len(documents)} pages of text.")

Found 5 PDFs:
 - Deep-Learning.pdf
 - Machine-Learning.pdf
 - NLP.pdf
 - RAG.pdf
 - Transformers.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/Widths': IndirectObject(17697, 0, 2175743530896), '/FirstChar': 0, '/LastChar': 188, '/BaseFont': '/DBFEAD+txsys', '/FontDescriptor': IndirectObject(17778, 0, 2175743530896)}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/Widths': IndirectObject(17697, 0, 2175743530896), '/FirstChar': 0, '/LastChar': 188, '/BaseFont': '/DBFEAD+txsys', '/FontDescriptor': IndirectObject(17778, 0, 2175743530896)}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/Widths': IndirectObject(17703, 0, 2175743530896), '/FirstChar': 116, '/LastC

Extracted 1823 pages of text.


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
metadatas = []

for doc in documents:
    for piece in splitter.split_text(doc["text"]):
        chunks.append(piece)
        metadatas.append({"source": doc["source"], "page": doc["page"]})

print(f"Created {len(chunks)} chunks.")

Created 8525 chunks.


In [7]:
# 1. Load dense embeddings (all-MiniLM-L6-v2, 384 dimensions)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Load or fit BM25 encoder
if os.path.exists("bm25_values.json"):
    bm25_encoder = BM25Encoder().load("bm25_values.json")
    print("Loaded existing BM25 encoder from bm25_values.json.")
else:
    bm25_encoder = BM25Encoder().default()
    bm25_encoder.fit(chunks)
    bm25_encoder.dump("bm25_values.json")
    print("BM25 encoder fitted and saved.")

# 3. Ensure index connection is initialized
index = pc.Index(INDEX_NAME)

# 4. Build Pinecone Hybrid Search Retriever
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index,
    top_k=5,
)

# 5. Upload chunks to Pinecone (only if index is empty)
stats = index.describe_index_stats()
current_count = stats.get("total_vector_count", 0)
if current_count == 0:
    print(f"Upserting {len(chunks)} chunks into Pinecone in batches of 100...")
    batch_size = 100
    for i in range(0, len(chunks), batch_size):
        retriever.add_texts(chunks[i:i+batch_size], metadatas=metadatas[i:i+batch_size])
        print(f"  Uploaded batch {i // batch_size + 1}/{(len(chunks) - 1) // batch_size + 1}")
    print("Chunks successfully indexed into Pinecone!")
else:
    print(f"Index already contains {current_count} vectors. Ready for retrieval!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2979.92it/s]


Loaded existing BM25 encoder from bm25_values.json.
Index already contains 8522 vectors. Ready for retrieval!


In [8]:
query = "What is Retrieval-Augmented Generation and what are its approaches?"
results = retriever.invoke(query)

print(f"Retrieved {len(results)} relevant chunks for query: '{query}'\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} [Source: {doc.metadata.get('source')}, Page: {doc.metadata.get('page')}] ---")
    print(doc.page_content[:250] + "...\n")

Retrieved 5 relevant chunks for query: 'What is Retrieval-Augmented Generation and what are its approaches?'

--- Result 1 [Source: RAG.pdf, Page: 2.0] ---
approaches
to
mitigate
the
LLM
hallucinations
such
as
fine-tuning,
prompt
engineering,
retrieval
augmented
generation
(RAG)
etc.
Retrieval
augmented
generation
(RAG)
has
been
the
most
talked
about
approach
in
mitigating
the
hallucinations
faced
by
la...

--- Result 2 [Source: RAG.pdf, Page: 29.0] ---
Created by Pavan Belagatti
RAG
Approaches
RAG
is
no
longer
just
about
retrieval-
it's
about
smart,
self-improving
intelligence!
We
were
all
so
excited
when
RAG
was
first
introduced.
We
still
are,
this
is
never
ending.
I
mean,
RAG
will
still
remain
re...

--- Result 3 [Source: RAG.pdf, Page: 3.0] ---
:
This
part
involves
enhancing
and
adding
more
relevant
context
to
the
retrieved
response
for
the
user
query.
●
Generation
:
Finally,
a
final
output
is
presented
to
the
user
with
the
help
of
a
large
language
model
(LLM).
The
LLM
uses
its
own

In [9]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.2,
)

context = "\n\n".join([d.page_content for d in results])

prompt = f"""You are an expert AI assistant. Answer the question using ONLY the provided context. If the context does not have enough information, state that clearly.

Context:
{context}

Question: {query}

Answer:"""

response = llm.invoke(prompt)
print("=== Gemini 2.5 Hybrid RAG Response ===\n")
print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


=== Gemini 2.5 Hybrid RAG Response ===

Retrieval-Augmented Generation (RAG) is a natural language processing framework that was first introduced by Meta AI researchers in 2020 through their paper "Retrieval-Augmented Generation for Knowledge-Intensive NLP Task" to address knowledge-intensive tasks. It involves enhancing and adding more relevant context to the retrieved response for a user query, and then a large language model (LLM) uses its own knowledge and the provided context to present a final output to the user.

Approaches to RAG mentioned in the context include:
*   **Multimodal Retrieval-Augmented Generation (MM-RAG)**: This extends the RAG concept by incorporating retrieval mechanisms to pull relevant information from external sources, processing and understanding data from multiple modalities like text, images, audio, and video.
*   **Agentic RAG**: This combines the power of RAG with autonomous agents, offering a more dynamic and context-aware method for information retrie